# MMIP Week7：多模態多輪客服、四種分流實驗與整合介面
先在 Colab 的「執行階段 → 變更執行階段類型」選 GPU。本筆記本已內含完整作業檔，無需另外上傳壓縮包。依序執行；所有模型訓練與推論都在 Colab，電腦本機只存放作業檔。

In [ ]:
from google.colab import files
from pathlib import Path
import base64
import io
import zipfile
payload = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
    archive.extractall('/content')
assert Path('/content/MMIP_Week7/requirements.txt').exists()
%cd /content/MMIP_Week7

## 1. 安裝環境並檢查 GPU

In [ ]:
%pip -q install -r requirements.txt
%pip uninstall -y torchao
import torch
assert torch.cuda.is_available(), '請先將 Colab 執行階段切換成 GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 2. 產生資料並執行不需 GPU 的控制流程驗證
已有 160 筆訓練樣本、20 筆獨立 Benchmark 與五張合成測試圖片。

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'make_data.py'], check=True)
subprocess.run([sys.executable, 'verify.py'], check=True)

## 3. 實驗 B：在 Colab GPU 訓練 LoRA 分流節點

In [ ]:
subprocess.run([sys.executable, 'train.py', '--epochs', '3'], check=True)

## 4. 實驗 A/B/C/D：實際圖片理解與量化比較
下列 USD 1/小時只是統一的計算成本基準，並非 Colab 帳單；若知道實際費率可自行修改。

In [ ]:
subprocess.run([sys.executable, 'benchmark.py', '--vision', 'model', '--gpu-hourly-usd', '1.0'], check=True)
from IPython.display import Markdown, display
display(Markdown(Path('artifacts/benchmark_model/results.md').read_text(encoding='utf-8')))

## 5. 下載訓練權重、逐題結果與報告

In [ ]:
import shutil
shutil.make_archive('/content/MMIP_Week7_results', 'zip', '/content/MMIP_Week7', 'artifacts')
files.download('/content/MMIP_Week7_results.zip')

## 6. 啟動聊天介面
執行後點開 Gradio 暫時連結。介面會在這個 Colab 執行階段運作；關閉執行階段後連結即失效。

In [ ]:
subprocess.run([sys.executable, 'app.py', '--host', '0.0.0.0', '--share'], check=True)